# 40 · Relate, allowing for spelling, against pandas, rapidfuzz and jellyfish

**Question.** Fieldwork experiment 193 let Relate match names spelled differently: folded, the words named dropped,
and a name not equal matched to the one spelled closest at or above a Jaro-Winkler threshold, 0.88 by default. On
Sierra Leone's 2019 measles cases by residential district ("Bo District") against OCHA COD-AB's district names ("Bo",
"Fabala"), does pandas fold every name the same way, reach every match, and build the same related table?

The folded names and the matches are exact; a spelling score is held to **1e-9** of rapidfuzz's.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "rapidfuzz", "jellyfish"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

Three cases: District dropped at the default 0.88; the same at 0.96, where Falaba's spelling (0.956) falls below;
and 0.88 with nothing dropped, where the shared prefix alone carries most names over the threshold.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_40_relate_spelling import run

report = run(Path.cwd() / "fixtures" / "relate-spelling-sl-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{'OK ' if c['agrees'] else 'DIFF'} case {c['case']}: {c['note']}")
    print(f"  outcomes {c['outcomes']}; {c['rows']} rows, {'equal' if c['rowsAgree'] else 'NOT equal'} to the pandas merge")
    for n in c["names"]:
        if n["fieldwork"]["outcome"] != "exact" or not n["agrees"]:
            p, f = n["pandas"], n["fieldwork"]
            print(f"  {'OK ' if n['agrees'] else 'DIFF'} {n['name']}: {p['outcome']} {p['value'] or ''} {p['score'] or ''}"
                  f" / {f['outcome']} {f['value'] or ''} {f['score'] or ''}")

## On the threshold

A score equal to the threshold passes. Four names in case 3 score exactly 0.88, where floating point on either side
could fall either way; the check names them, and both sides passed them.

In [ ]:
print("on the threshold:", [f"case {t['case']}: {t['name']}" for t in report["onThreshold"]])
print("jellyfish differs from rapidfuzz:", report["jellyfishDiffers"] or "nowhere")

## What this cannot establish

**That a spelling match is the same place.** "Fabala" is Falaba because COD-AB holds no other district like it,
which a person sees in Relate's preview and the step cannot. **That 0.88 suits district names**: case 3 shows the
words dropped do more of the matching than the threshold does.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"], "· rapidfuzz", report["environment"]["rapidfuzz"], "· jellyfish", report["environment"]["jellyfish"])
print("runtime :", report["environment"]["runtime"])